# 데이터베이스 다루기 : PyMysql 사용

In [1]:
# 설치
# pip install pymysql
# pip install cryptography 

In [2]:
# 기본적인 작업 흐름
# 1. Connection 생성 ← DB 서버 접근하기 위해 필요
# 2. 위 Connection 으로부터 Cursor 객체 생성 ← DB 다루기 위해 필요
# 3. Cursor 객체를 사용하여
#    - DDL , DML 명령 실행 -> executeXXX()
#    - SELECT 명령 실행 => fetchXXX()

# 4. Cursor, Connection 객체는 시스템 자원이기 때문에 
#    사용후에는 반드시 close() 해준다


## 기본 흐름 Connection, Cursor

In [3]:
import pymysql

In [4]:
conn = pymysql.connect(
    host='localhost',  # 접속url
    # port=3306,     # 디폴트 3306
    user='user2604',
    password='1234',
    database='db2604',
    # autocommit=False, # 디폴트 False
)


print(type(conn))
# connection 생성후 Cursor 객체 생성
cursor = conn.cursor()
print(type(cursor))

# Cursor 자원 반납
cursor.close()

# conneciton 자원 반납
conn.close() 

<class 'pymysql.connections.Connection'>
<class 'pymysql.cursors.Cursor'>


## with, try, transaction

In [5]:
# pymysql은 기본적으로 connection 객체는 context manager(with)를 지원하지만, 
# cursor는 구버전에서는 직접 지원이 없을 수 있다. (이 경우 보통 contextlib.closing을 같이 사용)


In [6]:
# 접속 정보 dict 로 준비
conn_info = {
    "host": "localhost",
    "user": "user2605",
    "password": '1234',
    "database": 'db2604',
}

"""
with pymysql.connect(**conn_info) as conn:
    with conn.cursor() as cursor:
        print(type(conn))
        print(type(cursor))
        # DB 관련 쿼리 실행...
"""        

# 🚩주의] with, close 관련
# Cursor는 with conn.cursor() as cursor: 구문으로 안전하게 닫히지만, 
# Connection은 with 구문이 끝나도 conneciton 이 닫히지 않습니다! 💢
# 반드시 finally 블록에서 '명시적'으로 close()를 호출해야 합니다.        

conn = None
try:
    conn = pymysql.connect(**conn_info)
    with conn.cursor() as cursor:
        print(type(conn))
        print(type(cursor))
        # 수행할 DB 작업들...

    # DML 작업의 경우 (INSERT, UPDATE, DELETE)
    conn.commit()

except Exception as e:
    conn and conn.rollback()  # 트랜잭션 중 오류 발생하면. rollback
    print('💥Error:', e)

finally:
    conn and conn.close()



💥Error: (1045, "Access denied for user 'user2605'@'192.168.65.1' (using password: YES)")


# DDL 데이터베이스 생성

In [7]:
conn_info = {
    "host": "localhost",
    "user": "root",
    "password": '1234',
}

conn = None
try:
    conn = pymysql.connect(**conn_info)
    with conn.cursor() as cursor:
        sql = 'CREATE DATABASE IF NOT EXISTS pydb'
        cursor.execute(sql)   # DDL 실행 execute()

except Exception as e:
    print("💥Error:", e)  # 예외의 [0] 에러코드,  [1] 에러메세지
finally:
    conn and conn.close()
    

# 테이블 있는 상태에서 실행하면 에러 나온다.
# 이때 에러는 ProgrammingError 이고
# 에러 메세지는 (1007, "Can't create database 'pydb'; database exists")
# 에러 메세지는 (MySQL에러코드, MySQL에러메세지) tuple 이다    

# DDL 테이블 생성

In [8]:
conn_info = {
    "host": "localhost",
    "user": "root",
    "password": '1234',
    "database": 'pydb',  # database 지정
}

conn = None
try:
    conn = pymysql.connect(**conn_info)
    with conn.cursor() as cursor:
        sql = """
        CREATE TABLE users (
            id INT AUTO_INCREMENT PRIMARY KEY,
            name VARCHAR(255) NOT NULL,
            age INT NOT NULL,
            created_at DATETIME NOT NULL
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8            
        """
        result = cursor.execute(sql)   # DDL 실행 execute()
        print('✅테이블 생성 성공', result)  # 성공하면 0 리턴..

except Exception as e:
    print("💥Error:", e)
finally:
    conn and conn.close()

💥Error: (1050, "Table 'users' already exists")


# DML: INSERT

In [9]:
from datetime import datetime, timedelta

conn = None
try:
    conn = pymysql.connect(**conn_info)
    with conn.cursor() as cursor:
        # 방법1
        sql = "INSERT INTO users (name, age, created_at) VALUES ('소리', 13, '2026-01-23 13:21:56')"
        result = cursor.execute(sql)   # DML -> 정수 리턴
        print(f'✅ {result} 개 row insert 성공')

        # # 방법2 Prepared statement 방식
        # sql = "INSERT INTO users (name, age, created_at) VALUES (%s, %s, %s)"
        # result = cursor.execute(sql, ('야옹이', 19, datetime(2004, 5, 8, 21, 19, 12)))
        # print(f'✅ {result} 개 row insert 성공')

        # # 여러개 insert 가능
        # now = datetime.now()
        # data = [
        #     ("두부", 5, now + timedelta(days=5)),
        #     ("로켓", 21, now - timedelta(days=2)),
        #     ("몬지", 2, now + timedelta(hours=1)),                
        # ]

        # result = cursor.executemany(sql, data)
        # print(f'✅ {result} 개 row insert 성공')

    
    conn.commit()  # DML 수행후 commit 하기

    # cursor 객체에는 쿼리 실행결과에 대한 정보들도 담고 있다
    print('\t', cursor.lastrowid, cursor.rowcount)
        
except Exception as e:
    conn and conn.rollback()
    print("💥Error:", e)

finally:
    conn and conn.close()
    

✅ 1 개 row insert 성공
	 13 1


# SELECT

In [10]:
conn_info = {
    "host": "localhost",
    "user": "root",
    "password": '1234',
    "database": 'pydb',
    "charset": 'utf8',
}    

In [11]:
conn = None
try:
    conn = pymysql.connect(**conn_info)
    with conn.cursor() as cursor:
        sql = 'SELECT * FROM users ORDER BY id DESC'
        cursor.execute(sql)  # executeXXX() 후에 fetchXXX() 로 읽어온다.

        while True:
            result = cursor.fetchone()  # 한 row씩 
            if not result: break  # 더 이상 fetch 해올 row 가 없으면 None 리턴
            print(result)   # tuple 리턴    

except Exception as e:
    print(f"💥Error:", e)
finally:
    conn and conn.close()


(13, '소리', 13, datetime.datetime(2026, 1, 23, 13, 21, 56))
(3, '야옹이', 19, datetime.datetime(2004, 5, 8, 21, 19, 12))
(2, '베리베리', 3, datetime.datetime(2026, 1, 23, 13, 21, 56))
(1, '둘리', 13, datetime.datetime(2026, 1, 23, 13, 21, 56))


In [12]:
conn = None
try:
    conn = pymysql.connect(**conn_info)
    with conn.cursor() as cursor:
        sql = 'SELECT * FROM users ORDER BY id DESC'
        cursor.execute(sql)  # executeXXX() 후에 fetchXXX() 로 읽어온다.

        result = cursor.fetchall()  # 전체 row 들을 리턴 -> Tuple[Tuple]
        print(result)

except Exception as e:
    print(f"💥Error:", e)
finally:
    conn and conn.close()


((13, '소리', 13, datetime.datetime(2026, 1, 23, 13, 21, 56)), (3, '야옹이', 19, datetime.datetime(2004, 5, 8, 21, 19, 12)), (2, '베리베리', 3, datetime.datetime(2026, 1, 23, 13, 21, 56)), (1, '둘리', 13, datetime.datetime(2026, 1, 23, 13, 21, 56)))


# SELECT - DictCursor 사용

In [13]:
# DictCursor 사용
# SELECT 결과를 dict 형태로 받아오려면, pymysql.cursors.DictCursor 를 사용해야 합니다. 
# 이 커서를 사용하면 각 행이 dict 형태로 반환되어, 컬럼 이름으로 접근할 수 있습니다.

In [14]:
conn_info = {
    "host": "localhost",
    "user": "root",
    "password": '1234',
    "database": 'pydb',
    "charset": 'utf8',
    "cursorclass": pymysql.cursors.DictCursor,   # fetchXXX() 결과를 Dict 로 받아옴.
}     

In [15]:
conn = None
try:
    conn = pymysql.connect(**conn_info)
    with conn.cursor() as cursor:
        sql = 'SELECT * FROM users ORDER BY id DESC'
        cursor.execute(sql)

        result = cursor.fetchall()  # 전체 row 들을 리턴 -> List[Dict]
        print(result)

        print()

        for row in result:
            # print(row)
            print(row['id'], row['name'], row['age'], row['created_at'])

except Exception as e:
    print(f"💥Error:", e)
finally:
    conn and conn.close()

[{'id': 13, 'name': '소리', 'age': 13, 'created_at': datetime.datetime(2026, 1, 23, 13, 21, 56)}, {'id': 3, 'name': '야옹이', 'age': 19, 'created_at': datetime.datetime(2004, 5, 8, 21, 19, 12)}, {'id': 2, 'name': '베리베리', 'age': 3, 'created_at': datetime.datetime(2026, 1, 23, 13, 21, 56)}, {'id': 1, 'name': '둘리', 'age': 13, 'created_at': datetime.datetime(2026, 1, 23, 13, 21, 56)}]

13 소리 13 2026-01-23 13:21:56
3 야옹이 19 2004-05-08 21:19:12
2 베리베리 3 2026-01-23 13:21:56
1 둘리 13 2026-01-23 13:21:56


In [16]:
conn = None
try:
    conn = pymysql.connect(**conn_info)
    with conn.cursor() as cursor:
        # SELECT 하는 컬럼에 별명을 붙인 경우 dict 에서 key 값도 이 별명으로 불러와야 한다.
        sql = 'SELECT id, name, age `나이`, age+10 `10년뒤` FROM users ORDER BY id DESC'
        cursor.execute(sql)

        result = cursor.fetchall()
        print(result)

        print()

        for row in result:
            print(row)
            print(row['id'], row['name'], row['나이'], row['10년뒤'])

except Exception as e:
    print(f"💥Error:", e)
finally:
    conn and conn.close()

[{'id': 13, 'name': '소리', '나이': 13, '10년뒤': 23}, {'id': 3, 'name': '야옹이', '나이': 19, '10년뒤': 29}, {'id': 2, 'name': '베리베리', '나이': 3, '10년뒤': 13}, {'id': 1, 'name': '둘리', '나이': 13, '10년뒤': 23}]

{'id': 13, 'name': '소리', '나이': 13, '10년뒤': 23}
13 소리 13 23
{'id': 3, 'name': '야옹이', '나이': 19, '10년뒤': 29}
3 야옹이 19 29
{'id': 2, 'name': '베리베리', '나이': 3, '10년뒤': 13}
2 베리베리 3 13
{'id': 1, 'name': '둘리', '나이': 13, '10년뒤': 23}
1 둘리 13 23


# DML: UPDATE

In [17]:
conn = None
try:
    conn = pymysql.connect(**conn_info)
    with conn.cursor() as cursor:
        sql = "UPDATE users SET name = '둘리' WHERE id = 1"
        cursor.execute(sql)

        # SQL 문자열을 단순 문자열 포맷팅하는 것은 비추. 
        new_name = '희준'
        sql = f"UPDATE users SET name = '{new_name}' WHERE id = 2"
        cursor.execute(sql)

        # 서식문자는
        sql = "UPDATE users SET name = %s WHERE id = %s"
        cursor.execute(sql, ('베리베리', 2))


        # 결과 확인
        sql = "SELECT * FROM users ORDER BY id DESC"
        cursor.execute(sql)
        for row in cursor.fetchall():
            print(row)
    
    conn.commit()

except Exception as e:
    conn and conn.rollback()
    print("💥Error:", e)

finally:
    conn and conn.close()

{'id': 13, 'name': '소리', 'age': 13, 'created_at': datetime.datetime(2026, 1, 23, 13, 21, 56)}
{'id': 3, 'name': '야옹이', 'age': 19, 'created_at': datetime.datetime(2004, 5, 8, 21, 19, 12)}
{'id': 2, 'name': '베리베리', 'age': 3, 'created_at': datetime.datetime(2026, 1, 23, 13, 21, 56)}
{'id': 1, 'name': '둘리', 'age': 13, 'created_at': datetime.datetime(2026, 1, 23, 13, 21, 56)}


# DML: DELETE

In [18]:
conn = None
try:
    conn = pymysql.connect(**conn_info)
    with conn.cursor() as cursor:
        sql = "DELETE FROM users WHERE id > %s"
        cursor.execute(sql, 3)  # 포맷 문자가 1개뿐이면 tuple 로 전달안해도 됨.
        print(cursor.rowcount, '개의 row(s) 가 delete 되었습니다')
       

        # 결과 확인
        sql = "SELECT * FROM users ORDER BY id DESC"
        cursor.execute(sql)
        for row in cursor.fetchall():
            print(row)
    
    conn.commit()

except Exception as e:
    conn and conn.rollback()
    print("💥Error:", e)

finally:
    conn and conn.close()

1 개의 row(s) 가 delete 되었습니다
{'id': 3, 'name': '야옹이', 'age': 19, 'created_at': datetime.datetime(2004, 5, 8, 21, 19, 12)}
{'id': 2, 'name': '베리베리', 'age': 3, 'created_at': datetime.datetime(2026, 1, 23, 13, 21, 56)}
{'id': 1, 'name': '둘리', 'age': 13, 'created_at': datetime.datetime(2026, 1, 23, 13, 21, 56)}
